# 02 Song Table

## Goal

Create unique song and artist tables from the cleaned UK chart history.

### Tasks

- Load cleaned chart history
- Validate imported dataset
- Create unique song table
- Create unique artist table
- Prepare entities for the relational data model

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# Load Dataset

project_path = Path.cwd().parent
interim_path = project_path / 'data' / 'interim'

chart_history = pd.read_csv(interim_path/'uk_chart_history_clean.csv')

In [3]:
chart_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 210882 entries, 0 to 210881
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Song            210882 non-null  str  
 1   Artist          210837 non-null  str  
 2   Position        210882 non-null  int64
 3   Last Week       210882 non-null  str  
 4   Peak            210882 non-null  int64
 5   Weeks on Chart  210882 non-null  int64
 6   Week            210882 non-null  str  
dtypes: int64(3), str(4)
memory usage: 11.3 MB


In [4]:
chart_history.head()

,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
0,SAVE YOUR LOVE,RENEE AND RENATO,1,LW:1,1,11,2 January 1983- 8 January 1983
1,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,LW:6,2,6,2 January 1983- 8 January 1983
2,A WINTER'S TALE,DAVID ESSEX,3,LW:7,3,5,2 January 1983- 8 January 1983
3,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,LW:8,4,9,2 January 1983- 8 January 1983
4,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983


In [5]:
# Validate Dataset

print(f'Missing values:\n{chart_history.isna().sum()}')
print(f'\nDuplicated values:\n{chart_history.duplicated().sum()}')

Missing values:
Song               0
Artist            45
Position           0
Last Week          0
Peak               0
Weeks on Chart     0
Week               0
dtype: int64

Duplicated values:
0


### Initial Dataset Validation

The cleaned chart history was successfully loaded from the interim data folder.

Validation confirms:

- Expected dataset dimensions.
- No remaining duplicate records.
- Missing values are limited to the `Artist` column.
- Missing artist information will be reviewed during entity creation.

## Create Song Table

Unique song–artist combinations are extracted from the cleaned chart history.

This table serves as the foundation for Spotify matching and the subsequent enrichment with song metadata.

In [6]:
# Create Song Table

songs = chart_history[['Song', 'Artist']].copy()

unique_songs = songs.drop_duplicates().reset_index(drop=True)
unique_songs.shape

(36661, 2)

In [7]:
unique_songs.info()

<class 'pandas.DataFrame'>
RangeIndex: 36661 entries, 0 to 36660
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Song    36661 non-null  str  
 1   Artist  36655 non-null  str  
dtypes: str(2)
memory usage: 573.0 KB


In [8]:
unique_songs.head()

,Song,Artist
0,SAVE YOUR LOVE,RENEE AND RENATO
1,YOU CAN'T HURRY LOVE,PHIL COLLINS
2,A WINTER'S TALE,DAVID ESSEX
3,BEST YEARS OF OUR LIVES,MODERN ROMANCE
4,OUR HOUSE,MADNESS


### Missing Artist Information

Six songs contain missing artist information.

These records are documented separately and excluded from the song table until the missing artist information has been resolved. As a result, no `song_id` is assigned to these records at this stage.

In [9]:
# Identify Missing Artist Information

missing_artist_songs = unique_songs[unique_songs['Artist'].isna()]
missing_artist_songs

,Song,Artist
20681,WHO'S THAT GIRL,NaN
22905,THE STETS,NaN
23058,SATISFACTION,NaN
25830,1 2 STEP,NaN
27772,TAMBOURINE,NaN
28596,ECUADOR,NaN


## Create Song Identifiers

Unique identifiers are assigned to all songs with complete song and artist information.

The identifier serves as the primary key for the song table and enables consistent references with chart statistics and additional metadata in later project stages.

In [10]:
# Assign Unique Song Identifiers

unique_songs = unique_songs[unique_songs['Artist'].notna()].reset_index(drop=True)

unique_songs['song_id'] = range(1, len(unique_songs) + 1)
unique_songs = unique_songs[['song_id', 'Song', 'Artist']]

unique_songs.head()

,song_id,Song,Artist
0,1,SAVE YOUR LOVE,RENEE AND RENATO
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS
2,3,A WINTER'S TALE,DAVID ESSEX
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE
4,5,OUR HOUSE,MADNESS


## Create Artist Table

A unique artist table is extracted from the song table.

Each row represents one distinct artist and serves as the foundation for the relational data model and future artist-level analyses.

In [11]:
artists = unique_songs[['Artist']].drop_duplicates().reset_index(drop=True)
artists.shape

(15150, 1)

In [12]:
artists.info()

<class 'pandas.DataFrame'>
RangeIndex: 15150 entries, 0 to 15149
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Artist  15150 non-null  str  
dtypes: str(1)
memory usage: 118.5 KB


In [13]:
artists.isna().sum()

Artist    0
dtype: int64

In [14]:
# Assign Unique Artist Identifiers

artists = artists.dropna().reset_index(drop=True)

artists['artist_id'] = range(1, len(artists) + 1)
artists = artists[['artist_id', 'Artist']]

print(artists.shape)
print(artists.head())

(15150, 2)
   artist_id            Artist
0          1  RENEE AND RENATO
1          2      PHIL COLLINS
2          3       DAVID ESSEX
3          4    MODERN ROMANCE
4          5           MADNESS


### Initial Entity Creation

Two core entities were extracted from the cleaned chart history:

- Songs
- Artists

These tables form the basis for the relational data model used in the following project stages.

Unique identifiers and table relationships will be introduced after the conceptual data model has been finalized.

Both entity tables were exported as intermediate datasets for the next processing steps.

In [15]:
interim_path = project_path / 'data' / 'interim'

unique_songs.to_csv(interim_path/'songs.csv', index=False)
artists.to_csv(interim_path/'artists.csv', index=False)